# Arctic Route RL — 반복 학습 (보상 자동조정)
**colab_train.ipynb 이후 단계용**  
학습 → 평가 → 보상 가중치 자동조정 → 재학습을 수렴할 때까지 반복

**사용법**: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택 후 전체 실행

In [ ]:
# ============================================================
# CELL 1: GPU 확인
# ============================================================
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    raise RuntimeError('GPU 없음 — 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 재실행')

In [ ]:
# ============================================================
# CELL 2: 패키지 설치
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3[extra]==2.3.2', 'gymnasium==0.29.1'])
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CELL 3: Google Drive 마운트
# ============================================================
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/arctic_rl'
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
print(f'Drive 준비: {DRIVE_DIR}')

In [ ]:
# ============================================================
# CELL 4: 기본 모듈 작성 (config / ship_dynamics / land_mask / reward)
# ============================================================
import os
os.makedirs('/content/arctic/modules', exist_ok=True)

# ---------- config.py ----------
with open('/content/arctic/modules/config.py', 'w') as f:
    f.write('''
from __future__ import annotations
from typing import Dict, List, Tuple

ROUTE_WAYPOINTS: Dict[str, List[Tuple[float, float]]] = {
    "NSR": [
        (35.10, 129.04), (41.78, 140.81), (45.65, 141.93),
        (52.00, 155.00), (63.00, 174.00), (65.77, 169.30),
        (71.00, 180.00), (73.50, 165.00), (76.00, 140.00),
        (77.60, 104.30), (76.00, 80.00),  (72.00, 55.00),
        (70.50, 30.00),  (62.00, 5.00),   (51.90, 4.50),
    ],
    "NWP": [
        (35.10, 129.04), (45.65, 141.93), (52.00, 155.00),
        (65.77, 169.30), (71.50, -156.00),(74.30, -118.00),
        (74.00, -95.00), (72.50, -80.00), (66.50, -61.00),
        (58.00, -45.00), (51.90, 4.50),
    ],
    "TSR": [
        (35.10, 129.04), (45.65, 141.93), (65.77, 169.30),
        (75.00, 180.00), (85.00, 160.00), (88.00, 0.00),
        (80.00, -10.00), (72.00, 0.00),   (51.90, 4.50),
    ],
}

MAX_SAFE_CONCENTRATION: Dict[str, float] = {
    "PC2": 0.95, "PC3": 0.9, "PC4": 0.8, "PC5": 0.7,
    "PC6": 0.6,  "PC7": 0.5, "IA Super": 0.7, "IA": 0.6,
    "IB": 0.5,   "IC": 0.4,  "None": 0.3,
}

ICE_CLASS_FACTORS: Dict[str, float] = {
    "PC2": 0.6, "PC3": 0.7, "PC4": 0.8, "PC5": 0.9,
    "PC6": 1.0, "PC7": 1.1, "IA Super": 0.8, "IA": 0.9,
    "IB": 1.0,  "IC": 1.1,  "None": 1.3,
}
''')

# ---------- rl_ship_dynamics.py ----------
with open('/content/arctic/modules/rl_ship_dynamics.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass

DEG2RAD = math.pi / 180.0
RAD2DEG = 180.0 / math.pi
KM_PER_DEG_LAT = 111.32
NM_TO_KM = 1.852
KNOTS_TO_KMS = NM_TO_KM / 3600.0

@dataclass
class ShipState:
    lon: float = 0.0
    lat: float = 0.0
    heading: float = 0.0
    speed_knots: float = 14.0
    target_speed: float = 14.0

@dataclass
class ShipParams:
    max_speed_knots: float = 15.0
    min_speed_knots: float = 3.0
    max_turn_rate_deg_s: float = 1.5
    speed_accel_knots_s: float = 0.02
    speed_decel_knots_s: float = 0.05
    turn_rate_speed_factor: float = 0.7
    ice_drag_factor: float = 0.4

def normalize_angle(deg):
    deg = deg % 360.0
    if deg > 180.0: deg -= 360.0
    return deg

def km_per_deg_lon(lat):
    return KM_PER_DEG_LAT * math.cos(lat * DEG2RAD)

def approx_dist_km(lat1, lon1, lat2, lon2):
    d_lat = (lat2 - lat1) * KM_PER_DEG_LAT
    d_lon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2.0)
    return math.sqrt(d_lat**2 + d_lon**2)

def bearing_deg(lat1, lon1, lat2, lon2):
    d_lon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2.0)
    d_lat = (lat2 - lat1) * KM_PER_DEG_LAT
    return normalize_angle(math.atan2(d_lon, d_lat) * RAD2DEG)

def step_ship(state, params, heading_delta_deg, speed_factor, ice_concentration, dt):
    ice_drag = 1.0 - params.ice_drag_factor * min(ice_concentration, 1.0)
    target = params.max_speed_knots * speed_factor * ice_drag
    target = max(params.min_speed_knots, min(params.max_speed_knots, target))
    speed = state.speed_knots
    if speed < target:
        speed = min(target, speed + params.speed_accel_knots_s * dt)
    else:
        speed = max(target, speed - params.speed_decel_knots_s * dt)
    speed_ratio = speed / params.max_speed_knots
    eff_turn = params.max_turn_rate_deg_s * (params.turn_rate_speed_factor + (1.0 - params.turn_rate_speed_factor) * speed_ratio)
    max_turn = eff_turn * dt
    actual_turn = max(-max_turn, min(max_turn, heading_delta_deg))
    heading = normalize_angle(state.heading + actual_turn)
    dist_km = speed * KNOTS_TO_KMS * dt
    heading_rad = heading * DEG2RAD
    d_lat = dist_km * math.cos(heading_rad) / KM_PER_DEG_LAT
    cos_lat = max(0.01, math.cos(state.lat * DEG2RAD))
    d_lon = dist_km * math.sin(heading_rad) / (KM_PER_DEG_LAT * cos_lat)
    lat = max(-89.9, min(89.9, state.lat + d_lat))
    lon = state.lon + d_lon
    if lon > 180.0: lon -= 360.0
    elif lon < -180.0: lon += 360.0
    return ShipState(lon=lon, lat=lat, heading=heading, speed_knots=speed, target_speed=target)
''')

# ---------- rl_land_mask.py ----------
with open('/content/arctic/modules/rl_land_mask.py', 'w') as f:
    f.write('''
from __future__ import annotations
_LAND_BOXES = [
    (59.0, 83.5, -73.0, -18.0),(63.0, 67.0, -25.0, -13.0),(74.0, 81.0, 10.0, 33.0),
    (70.0, 77.5, 51.0, 68.5),(78.0, 81.5, 91.0, 107.0),(79.5, 82.0, 44.0, 65.0),
    (73.0, 76.5, 136.0, 159.0),(70.5, 71.5, -179.0, -178.0),(70.5, 71.5, 178.5, 180.0),
    (72.0, 84.0, -90.0, -61.0),(68.0, 74.0, -115.0, -95.0),(65.0, 71.5, -165.0, -145.0),
    (58.0, 71.5, 5.0, 28.0),(73.0, 77.5, 92.0, 106.0),(69.5, 73.0, 67.0, 73.0),
    (64.5, 67.5, 172.0, 180.0),(51.5, 59.0, 160.5, 163.0),(43.5, 45.5, 141.5, 145.0),
    (46.0, 54.0, 141.8, 143.2),
]
_WAYPOINT_WHITELIST = [
    (45.65, 141.93, 0.5),(41.78, 140.81, 0.5),(74.30, -118.00, 1.5),
    (72.50, -80.00, 1.5),(66.50, -61.00, 1.5),(71.00, 180.00, 1.0),
    (71.00, -180.00, 1.0),(71.50, -156.00, 1.5),(65.77, 169.30, 0.5),
    (77.60, 104.30, 1.5),(72.00, 55.00, 1.5),(76.00, 80.00, 1.5),
    (76.00, 140.00, 1.5),(73.50, 165.00, 1.5),(58.00, -45.00, 1.0),
    (62.00, 5.00, 1.5),(74.00, -95.00, 1.5),
]
class LandMask:
    def is_land(self, lat, lon):
        lon_n = ((lon + 180.0) % 360.0) - 180.0
        for wp_lat, wp_lon, r in _WAYPOINT_WHITELIST:
            if abs(lat - wp_lat) <= r and abs(lon_n - wp_lon) <= r: return False
        for la, lb, lo, lx in _LAND_BOXES:
            if la <= lat <= lb and lo <= lon_n <= lx: return True
        return False
''')

# ---------- rl_reward.py ----------
with open('/content/arctic/modules/rl_reward.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass
from .config import MAX_SAFE_CONCENTRATION, ICE_CLASS_FACTORS

@dataclass
class RewardWeights:
    collision: float = -50.0
    proximity: float = -0.5
    danger_zone: float = -1.0
    route_deviation: float = -0.5
    progress: float = 10.0
    smoothness: float = -0.01
    fuel: float = -0.005
    ice_concentration: float = -0.1
    episode_success: float = 200.0

@dataclass
class RewardContext:
    ship_lat: float; ship_lon: float; ship_speed_knots: float
    heading_change_deg: float; speed_factor: float
    iceberg_distances_km: list; iceberg_sizes_m: list
    cross_track_error_km: float; along_track_progress: float
    max_allowed_deviation_km: float = 30.0
    ice_concentration: float = 0.0; max_safe_concentration: float = 0.7
    visibility_km: float = 10.0; wave_height_m: float = 1.0
    collision: bool = False; episode_done_success: bool = False

def compute_dynamic_safety_radius(base_radius_km, speed_knots, visibility_km, ice_class_factor=1.0):
    return base_radius_km * max(0.5, speed_knots/12.0) * (1.0 + 1.0/max(visibility_km,1.0)) * ice_class_factor

def compute_reward(ctx, weights=None):
    if weights is None: weights = RewardWeights()
    c = {}
    c["collision"] = weights.collision if ctx.collision else 0.0
    sr = compute_dynamic_safety_radius(10.0, ctx.ship_speed_knots, ctx.visibility_km)
    prox = 0.0; dz = 0.0
    for i, d in enumerate(ctx.iceberg_distances_km):
        sz = ctx.iceberg_sizes_m[i] if i < len(ctx.iceberg_sizes_m) else 5000.0
        sf = min(2.0, sz/5000.0); cr = max(0.5, sz/1000.0/2.0)
        if d < cr*2.0: dz = dz + float(sf*(1.0 - d/(cr*2.0)))
        elif d < sr*3: prox = prox + float(math.exp(-(d/sr)**2)*sf)
    c["proximity"] = weights.proximity * prox
    c["danger_zone"] = weights.danger_zone * dz
    c["route_deviation"] = weights.route_deviation * min(1.0, abs(ctx.cross_track_error_km)/ctx.max_allowed_deviation_km)
    if ctx.along_track_progress > 1e-5: c["progress"] = weights.progress
    elif ctx.along_track_progress < -1e-5: c["progress"] = weights.progress * 0.5
    else: c["progress"] = 0.0
    c["smoothness"] = weights.smoothness * min(1.0, abs(ctx.heading_change_deg)/15.0)
    c["fuel"] = weights.fuel * (1.0 - ctx.speed_factor)
    c["ice_concentration"] = weights.ice_concentration * min(1.0, ctx.ice_concentration/max(ctx.max_safe_concentration,1e-6)) if ctx.max_safe_concentration > 0 else 0.0
    c["episode_success"] = weights.episode_success if ctx.episode_done_success else 0.0
    return sum(c.values()), c
''')

with open('/content/arctic/modules/__init__.py', 'w') as f:
    f.write('')
print('기본 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 5: 환경 모듈 작성
# ============================================================
with open('/content/arctic/modules/rl_environment.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math, random
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from .rl_ship_dynamics import (ShipState, ShipParams, step_ship, approx_dist_km,
    bearing_deg, normalize_angle, KM_PER_DEG_LAT, km_per_deg_lon)
from .rl_reward import RewardContext, RewardWeights, compute_reward, compute_dynamic_safety_radius
from .rl_land_mask import LandMask
from .config import ROUTE_WAYPOINTS, MAX_SAFE_CONCENTRATION

class Iceberg:
    __slots__ = ("lat","lon","length_m","width_m")
    def __init__(self, lat, lon, length_m=5000.0, width_m=3000.0):
        self.lat=lat; self.lon=lon; self.length_m=length_m; self.width_m=width_m

def _random_icebergs(lat1,lon1,lat2,lon2,count,spread_km=35.0):
    bergs=[]
    for _ in range(count):
        t=random.random(); cl=lat1+t*(lat2-lat1); co=lon1+t*(lon2-lon1)
        ol=random.gauss(0,spread_km/KM_PER_DEG_LAT/3)
        oo=random.gauss(0,spread_km/max(1,km_per_deg_lon(cl))/3)
        st=random.choices(["small","medium","large","tabular"],weights=[0.4,0.3,0.2,0.1])[0]
        sz={"small":(random.uniform(25,80),random.uniform(15,50)),
            "medium":(random.uniform(80,200),random.uniform(50,120)),
            "large":(random.uniform(200,500),random.uniform(100,300)),
            "tabular":(random.uniform(500,2000),random.uniform(300,1000))}
        lm,wm=sz[st]; bergs.append(Iceberg(lat=cl+ol,lon=co+oo,length_m=lm,width_m=wm))
    return bergs

def _xt(slat,slon,wp1,wp2):
    la,lo=wp1; lb,lx=wp2
    An=(lb-la)*KM_PER_DEG_LAT; Ae=(lx-lo)*km_per_deg_lon((la+lb)/2)
    Pn=(slat-la)*KM_PER_DEG_LAT; Pe=(slon-lo)*km_per_deg_lon((la+slat)/2)
    AL=math.sqrt(An**2+Ae**2)
    if AL<1e-6: return approx_dist_km(slat,slon,la,lo)
    return (Pe*An-Pn*Ae)/AL

def _at(slat,slon,wp1,wp2):
    la,lo=wp1; lb,lx=wp2
    An=(lb-la)*KM_PER_DEG_LAT; Ae=(lx-lo)*km_per_deg_lon((la+lb)/2)
    Pn=(slat-la)*KM_PER_DEG_LAT; Pe=(slon-lo)*km_per_deg_lon((la+slat)/2)
    AB2=An**2+Ae**2
    if AB2<1e-6: return 0.0
    return max(0.0,min(1.0,(Pn*An+Pe*Ae)/AB2))

class IcebergAvoidanceEnv(gym.Env):
    metadata={"render_modes":["human"]}
    MAX_STEPS=3000; DT=2.0; MAX_DEVIATION_KM=50.0
    COLLISION_RADIUS_KM=0.5; MAX_NEARBY_ICEBERGS=3
    SEGMENT_MAX_DIST_KM=40.0; SUCCESS_PROGRESS=0.90

    def __init__(self,render_mode=None,difficulty="medium",reward_weights=None,
                 fixed_route=None,fixed_ice_class=None,ship_params=None):
        super().__init__()
        self.render_mode=render_mode; self.difficulty=difficulty
        self._fixed_route=fixed_route; self._fixed_ice_class=fixed_ice_class
        self._custom_ship_params=ship_params
        self.action_space=spaces.Box(low=np.array([-15.0,0.5],dtype=np.float32),high=np.array([15.0,1.0],dtype=np.float32))
        self.observation_space=spaces.Box(low=-np.ones(22,dtype=np.float32)*2.0,high=np.ones(22,dtype=np.float32)*2.0)
        self.ship=None; self.ship_params=ship_params if ship_params else ShipParams()
        self.reward_weights=reward_weights if reward_weights else RewardWeights()
        self.icebergs=[]; self.route_wps=[]
        self.segment_start_idx=0; self.segment_end_idx=0
        self.ice_class="PC5"; self.max_safe_conc=0.7
        self.ice_concentration=0.0; self.visibility_km=10.0
        self.wave_height_m=1.0; self.step_count=0; self.prev_progress=0.0
        self.land_mask=LandMask()

    def _dp(self):
        if self.difficulty=="easy": return {"bc":(0,0),"ic":(0.0,0.05),"vis":(18,20),"wv":(0.0,0.3)}
        elif self.difficulty=="medium": return {"bc":(3,8),"ic":(0.1,0.3),"vis":(8,15),"wv":(0.5,1.5)}
        else: return {"bc":(8,20),"ic":(0.2,0.6),"vis":(3,8),"wv":(1.0,3.0)}

    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        rk=self._fixed_route if self._fixed_route else random.choice(list(ROUTE_WAYPOINTS.keys()))
        self.route_wps=ROUTE_WAYPOINTS[rk]; n=len(self.route_wps)
        for _ in range(20):
            sl=random.randint(1,min(3,n-1)); si=random.randint(0,n-sl-1); ei=si+sl
            sd=sum(approx_dist_km(self.route_wps[i][0],self.route_wps[i][1],
                                  self.route_wps[i+1][0],self.route_wps[i+1][1]) for i in range(si,ei))
            if sd<=self.SEGMENT_MAX_DIST_KM: break
        self.segment_start_idx=si; self.segment_end_idx=ei
        self.ice_class=self._fixed_ice_class if self._fixed_ice_class else random.choice(["PC3","PC5","PC7","IA Super","IA"])
        self.max_safe_conc=MAX_SAFE_CONCENTRATION.get(self.ice_class,0.7)
        dp=self._dp(); bc=random.randint(*dp["bc"])
        self.ice_concentration=random.uniform(*dp["ic"])
        self.visibility_km=random.uniform(*dp["vis"])
        self.wave_height_m=random.uniform(*dp["wv"])
        self.icebergs=[]
        if bc>0:
            for i in range(si,ei):
                c=max(1,bc//(ei-si))
                self.icebergs.extend(_random_icebergs(self.route_wps[i][0],self.route_wps[i][1],
                                                      self.route_wps[i+1][0],self.route_wps[i+1][1],c))
        swp=self.route_wps[si]; nwp=self.route_wps[si+1]
        self.ship=ShipState(lon=swp[1],lat=swp[0],heading=bearing_deg(swp[0],swp[1],nwp[0],nwp[1]),speed_knots=14.0,target_speed=14.0)
        self.step_count=0; self.prev_progress=0.0
        return self._obs(),{}

    def _progress(self):
        sd=[]; td=0.0
        for i in range(self.segment_start_idx,self.segment_end_idx):
            d=float(approx_dist_km(self.route_wps[i][0],self.route_wps[i][1],self.route_wps[i+1][0],self.route_wps[i+1][1]))
            sd.append(d); td+=d
        if td<1e-3: return 0.0
        bc=0.0; cb=0.0
        for k,d in enumerate(sd):
            i=self.segment_start_idx+k
            f=float(_at(self.ship.lat,self.ship.lon,self.route_wps[i],self.route_wps[i+1]))
            cand=cb+f*d
            if cand>bc: bc=cand
            if f<1.0: break
            cb+=d
        return min(1.0,float(bc)/float(td))

    def _cross(self):
        bx=float("inf")
        for i in range(self.segment_start_idx,self.segment_end_idx):
            x=_xt(self.ship.lat,self.ship.lon,self.route_wps[i],self.route_wps[i+1])
            if abs(x)<abs(bx): bx=x
        return bx

    def _near(self,n=3):
        ds=[(normalize_angle(bearing_deg(self.ship.lat,self.ship.lon,b.lat,b.lon)-self.ship.heading),
             approx_dist_km(self.ship.lat,self.ship.lon,b.lat,b.lon),b.length_m) for b in self.icebergs]
        ds.sort(key=lambda x:x[1]); r=ds[:n]
        while len(r)<n: r.append((0.0,999.0,0.0))
        return r

    def _obs(self):
        o=np.zeros(22,dtype=np.float32)
        o[0]=self.ship.lon/180.0; o[1]=self.ship.lat/90.0
        hr=self.ship.heading*math.pi/180.0; o[2]=math.sin(hr); o[3]=math.cos(hr)
        o[4]=self.ship.speed_knots/self.ship_params.max_speed_knots
        ti=self.segment_start_idx+1
        for i in range(self.segment_start_idx+1,self.segment_end_idx+1):
            if _at(self.ship.lat,self.ship.lon,self.route_wps[i-1],self.route_wps[i])<0.95: ti=i; break
        tw=self.route_wps[min(ti,len(self.route_wps)-1)]
        o[5]=(tw[1]-self.ship.lon)*km_per_deg_lon(self.ship.lat)/100.0
        o[6]=(tw[0]-self.ship.lat)*KM_PER_DEG_LAT/100.0
        o[7]=normalize_angle(bearing_deg(self.ship.lat,self.ship.lon,tw[0],tw[1])-self.ship.heading)/180.0
        o[8]=min(1.0,approx_dist_km(self.ship.lat,self.ship.lon,tw[0],tw[1])/200.0)
        for i,(rb,d,sz) in enumerate(self._near(self.MAX_NEARBY_ICEBERGS)):
            o[9+i*2]=rb/180.0; o[10+i*2]=min(1.0,d/50.0)
        o[15]=min(1.0,self.ice_concentration); o[16]=min(1.0,self.max_safe_conc)
        o[17]=min(1.0,self.visibility_km/20.0); o[18]=min(1.0,self.wave_height_m/8.0)
        o[19]=self.max_safe_conc; o[20]=self._progress()
        o[21]=max(-1.0,min(1.0,self._cross()/self.MAX_DEVIATION_KM))
        return o

    def step(self,action):
        hd=float(np.clip(action[0],-15.0,15.0)); sf=float(np.clip(action[1],0.5,1.0))
        self.ship=step_ship(self.ship,self.ship_params,hd,sf,self.ice_concentration,self.DT)
        self.step_count+=1
        col=False; id_=[]; is_=[]
        for b in self.icebergs:
            d=approx_dist_km(self.ship.lat,self.ship.lon,b.lat,b.lon)
            id_.append(d); is_.append(b.length_m)
            if d<max(self.COLLISION_RADIUS_KM,b.length_m/1000.0/2.0): col=True
        if not col and self.land_mask.is_land(self.ship.lat,self.ship.lon): col=True
        cp=self._progress(); dp=cp-self.prev_progress; self.prev_progress=cp
        xk=abs(self._cross())
        term=False; trunc=False; succ=False
        if col: term=True
        elif cp>=self.SUCCESS_PROGRESS: term=True; succ=True
        elif xk>self.MAX_DEVIATION_KM: term=True
        elif self.step_count>=self.MAX_STEPS: trunc=True
        ctx=RewardContext(ship_lat=self.ship.lat,ship_lon=self.ship.lon,ship_speed_knots=self.ship.speed_knots,
            heading_change_deg=hd,speed_factor=sf,iceberg_distances_km=id_,iceberg_sizes_m=is_,
            cross_track_error_km=xk,along_track_progress=dp,max_allowed_deviation_km=self.MAX_DEVIATION_KM,
            ice_concentration=self.ice_concentration,max_safe_concentration=self.max_safe_conc,
            visibility_km=self.visibility_km,wave_height_m=self.wave_height_m,collision=col,episode_done_success=succ)
        rw,rc=compute_reward(ctx,self.reward_weights)
        return self._obs(),rw,term,trunc,{"collision":col,"success":succ,"progress":cp,"cross_track_km":xk,"reward_components":rc,"step":self.step_count}
''')
print('환경 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 6: 에이전트 모듈 작성
# ============================================================
import sys
sys.path.insert(0, '/content/arctic')

with open('/content/arctic/modules/rl_agent.py', 'w') as f:
    f.write('''
from __future__ import annotations
import logging
from collections import deque
from itertools import islice
from pathlib import Path
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from gymnasium.wrappers import RecordEpisodeStatistics
from .rl_environment import IcebergAvoidanceEnv
from .rl_reward import RewardWeights

class _StopTraining(Exception): pass
logger = logging.getLogger(__name__)

DEFAULT_HP = {
    "learning_rate": 3e-4, "buffer_size": 300_000, "batch_size": 256,
    "gamma": 0.95, "tau": 0.005, "ent_coef": "auto",
    "train_freq": 1, "gradient_steps": 1, "learning_starts": 1_000,
    "policy_kwargs": {"net_arch": [256, 256]},
}

class MetricsCallback(BaseCallback):
    def __init__(self, log_interval=5000, verbose=0):
        super().__init__(verbose)
        self.log_interval = log_interval
        self.ep_rewards = deque(maxlen=2000)
        self.collisions = 0; self.successes = 0; self.total_ep = 0
        self.history = deque(maxlen=500)
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.total_ep += 1
                self.ep_rewards.append(float(info["episode"]["r"]))
            if info.get("collision"): self.collisions += 1
            if info.get("success"): self.successes += 1
        if self.num_timesteps % self.log_interval == 0 and self.total_ep > 0:
            recent = list(islice(reversed(self.ep_rewards), 100))
            m = {"timestep": self.num_timesteps, "episodes": self.total_ep,
                 "mean_reward_100": float(np.mean(recent)) if recent else 0.0,
                 "collision_rate": self.collisions / max(1, self.total_ep),
                 "success_rate": self.successes / max(1, self.total_ep)}
            self.history.append(m)
            logger.info(f"[RL] Step {m[\'timestep\']}: reward={m[\'mean_reward_100\']:.1f}, "
                        f"collision={m[\'collision_rate\']:.3f}, success={m[\'success_rate\']:.3f}")
        return True
    def latest(self):
        return self.history[-1] if self.history else {"timestep":0,"episodes":0,"mean_reward_100":0,"collision_rate":0,"success_rate":0}

class IcebergAvoidanceAgent:
    def __init__(self, hyperparams=None, model_key="default", model_base_dir="/content/arctic/models"):
        self.hp = {**DEFAULT_HP, **(hyperparams or {})}
        self.model_key = model_key
        self.model_dir = Path(model_base_dir) / f"sac_{model_key}"
        self.model_dir.mkdir(parents=True, exist_ok=True)
        self.model = None; self.env = None
        self.cb = MetricsCallback(); self._ver = 0

    def create_env(self, difficulty="medium", reward_weights=None, fixed_route=None, fixed_ice_class=None, ship_params=None):
        raw = IcebergAvoidanceEnv(difficulty=difficulty, reward_weights=reward_weights,
                                   fixed_route=fixed_route, fixed_ice_class=fixed_ice_class, ship_params=ship_params)
        self.env = RecordEpisodeStatistics(raw)
        return self.env

    def build_model(self, difficulty="medium", reward_weights=None):
        if self.env is None: self.create_env(difficulty, reward_weights=reward_weights)
        hp = self.hp
        self.model = SAC("MlpPolicy", self.env,
            learning_rate=hp["learning_rate"], buffer_size=hp["buffer_size"],
            batch_size=hp["batch_size"], gamma=hp["gamma"], tau=hp["tau"],
            ent_coef=hp["ent_coef"], train_freq=hp["train_freq"],
            gradient_steps=hp["gradient_steps"], learning_starts=hp["learning_starts"],
            policy_kwargs=hp["policy_kwargs"], verbose=1, device="auto")
        return self.model

    def train(self, total_timesteps=500_000, extra_cb=None):
        if self.model is None: self.build_model()
        self.cb = MetricsCallback(log_interval=5000)
        from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback
        ckpt = CheckpointCallback(save_freq=10_000, save_path=str(self.model_dir/"checkpoints"), name_prefix="sac_ckpt", verbose=0)
        cbs = [self.cb, ckpt] + ([extra_cb] if extra_cb else [])
        try:
            self.model.learn(total_timesteps=total_timesteps, callback=CallbackList(cbs), progress_bar=True)
        except _StopTraining:
            logger.info("[RL] 중단")
        finally:
            self._ver += 1; self.save()
        return self.cb.latest()

    def predict(self, obs, deterministic=True):
        return self.model.predict(obs, deterministic=deterministic)[0], 0.0

    def save(self, path=None):
        if self.model is None: return
        p = path or str(self.model_dir/f"sac_v{self._ver}")
        self.model.save(p)

    def load(self, path=None):
        if path: lp = path
        else:
            vs = sorted(self.model_dir.glob("sac_v*.zip"))
            if not vs: return False
            lp = str(vs[-1]).replace(".zip","")
        try:
            if self.env is None: self.create_env()
            self.model = SAC.load(lp, env=self.env, device="auto")
            return True
        except Exception as e:
            logger.error(f"[RL] 로드 실패: {e}"); return False
''')
print('에이전트 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 7: 트레이너 + RewardAdjuster + IterativeTrainer 작성
# ============================================================
with open('/content/arctic/modules/rl_trainer.py', 'w') as f:
    f.write('''
from __future__ import annotations
import dataclasses, json, logging, math, os, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
from stable_baselines3.common.callbacks import BaseCallback
from .rl_agent import IcebergAvoidanceAgent, _StopTraining
from .rl_environment import IcebergAvoidanceEnv
from .rl_reward import RewardWeights

logger = logging.getLogger(__name__)

class _StopCb(BaseCallback):
    def __init__(self, trainer):
        super().__init__(verbose=0); self.trainer=trainer
    def _on_step(self):
        if self.trainer.stop_requested: raise _StopTraining("중단")
        return True

@dataclass
class CurriculumStage:
    name: str; difficulty: str; timesteps: int; description: str

CURRICULUM = [
    CurriculumStage("stage_1_basic",    "easy",   50, "빙산 없음"),
    CurriculumStage("stage_2_moderate", "medium", 33, "빙산 도입"),
    CurriculumStage("stage_3_hard",     "hard",   17, "고난이도"),
]

# ── 보상 가중치 clamping 범위 ─────────────────────────────
WEIGHT_BOUNDS = {
    "collision":         (-1500.0, -50.0),
    "proximity":         (-30.0,   -0.5),
    "danger_zone":       (-50.0,   -1.0),
    "route_deviation":   (-2.0,    -0.01),
    "progress":          (0.5,     100.0),
    "smoothness":        (-0.5,    -0.01),
    "fuel":              (-0.2,    -0.005),
    "ice_concentration": (-5.0,    -0.1),
    "episode_success":   (100.0,   2000.0),
}

def _clamp(v, lo, hi): return max(lo, min(hi, v))

def _apply_bounds(w):
    d = dataclasses.asdict(w)
    for k,(lo,hi) in WEIGHT_BOUNDS.items():
        if k in d: d[k] = _clamp(d[k], lo, hi)
    return RewardWeights(**d)

def _clean_nan(obj):
    if isinstance(obj, float) and math.isnan(obj): return 0.0
    if isinstance(obj, dict): return {k:_clean_nan(v) for k,v in obj.items()}
    if isinstance(obj, list): return [_clean_nan(v) for v in obj]
    return obj


class RewardAdjuster:
    """평가 메트릭 → 시그널 분석 → 보상 가중치 자동 조정"""

    def analyze(self, metrics):
        signals = []
        cr = metrics.get("collision_rate", 1.0)
        sr = metrics.get("success_rate", 0.0)
        dev = metrics.get("mean_max_deviation_km", 0.0)
        if cr > 0.20: signals.append("critical_collision")
        elif cr > 0.10: signals.append("high_collision")
        if sr < 0.60: signals.append("low_success")
        elif sr < 0.70: signals.append("moderate_success")
        if dev > 30.0: signals.append("high_deviation")
        if cr <= 0.05 and sr >= 0.70: signals.append("converging")
        priority = ["critical_collision","high_collision","low_success",
                    "moderate_success","high_deviation","converging"]
        return [s for s in priority if s in signals][:3]

    def adjust(self, weights, signals):
        d = dataclasses.asdict(weights)
        for sig in signals:
            if sig == "critical_collision":
                d["collision"] *= 2.0; d["proximity"] *= 1.8; d["danger_zone"] *= 2.0
            elif sig == "high_collision":
                d["collision"] *= 1.6; d["proximity"] *= 1.4; d["danger_zone"] *= 1.5
            elif sig == "low_success":
                d["episode_success"] *= 1.8; d["progress"] *= 1.5
            elif sig == "moderate_success":
                d["episode_success"] *= 1.3; d["progress"] *= 1.2
            elif sig == "high_deviation":
                d["route_deviation"] *= 1.5; d["progress"] *= 0.85
            elif sig == "converging":
                d["smoothness"] *= 0.9; d["fuel"] *= 0.9
        return _apply_bounds(RewardWeights(**d))

    def check_plateau(self, history, field="collision_rate", threshold=0.02, window=3):
        if len(history) < window + 1: return False
        imps = [abs(history[-(i+1)]["post_metrics"].get(field,0) -
                    history[-(i+2)]["post_metrics"].get(field,0)) for i in range(window)]
        return all(imp < threshold for imp in imps)


@dataclass
class IterationRecord:
    iteration: int; weights: dict; pre_metrics: dict; post_metrics: dict
    signals: list; duration_seconds: float; converged: bool


class RLTrainer:
    def __init__(self, hyperparams=None, model_key="default", fixed_route=None,
                 fixed_ice_class=None, ship_params=None, model_base_dir="/content/arctic/models"):
        self.agent = IcebergAvoidanceAgent(hyperparams, model_key=model_key, model_base_dir=model_base_dir)
        self._fixed_route = fixed_route; self._fixed_ice_class = fixed_ice_class
        self._ship_params = ship_params; self.stop_requested = False
        self.is_training = False; self.current_stage = None; self.training_log = []

    def _create_env(self, difficulty, reward_weights=None):
        return self.agent.create_env(difficulty=difficulty, reward_weights=reward_weights,
            fixed_route=self._fixed_route, fixed_ice_class=self._fixed_ice_class, ship_params=self._ship_params)

    def train_curriculum(self, stages=None, reward_weights=None, base_timesteps=None):
        stages = stages or CURRICULUM
        self.is_training = True; self.stop_requested = False; results = []
        total_ratio = sum(s.timesteps for s in stages)
        def _ts(s):
            return s.timesteps if base_timesteps is None else max(10_000, int(base_timesteps*s.timesteps/total_ratio))
        try:
            for i, stage in enumerate(stages):
                if self.stop_requested: break
                self.current_stage = stage.name
                ts = _ts(stage)
                logger.info(f"[Trainer] 커리큘럼 {i+1}/{len(stages)}: {stage.name} ({ts:,} steps)")
                self._create_env(stage.difficulty, reward_weights)
                if self.agent.model is None:
                    self.agent.build_model(difficulty=stage.difficulty, reward_weights=reward_weights)
                else:
                    self.agent.model.set_env(self.agent.env)
                t0 = time.time()
                metrics = self.agent.train(total_timesteps=ts, extra_cb=_StopCb(self))
                results.append({"stage":stage.name,"difficulty":stage.difficulty,
                                "timesteps":ts,"elapsed":time.time()-t0,"metrics":metrics})
                if self.stop_requested: break
        finally:
            self.is_training = False; self.current_stage = None
        return {"stages": results}

    def evaluate(self, n_episodes=50, difficulty="medium"):
        import numpy as np
        if self.agent.model is None:
            if not self.agent.load(): return {"error":"모델 없음"}
        env = IcebergAvoidanceEnv(difficulty=difficulty, fixed_route=self._fixed_route,
            fixed_ice_class=self._fixed_ice_class, ship_params=self._ship_params)
        rewards=[]; collisions=0; successes=0
        for _ in range(n_episodes):
            obs, _ = env.reset(); total_r = 0
            while True:
                action, _ = self.agent.predict(obs, deterministic=True)
                obs, r, term, trunc, info = env.step(action)
                total_r += r
                if term or trunc:
                    if info.get("collision"): collisions += 1
                    if info.get("success"): successes += 1
                    break
            rewards.append(total_r)
        return {
            "episodes": len(rewards), "difficulty": difficulty,
            "mean_reward": float(np.mean(rewards)) if rewards else 0.0,
            "collision_rate": collisions/n_episodes,
            "success_rate": successes/n_episodes,
        }


class IterativeTrainer:
    """학습 → 평가 → 보상 가중치 자동조정 → 재학습 루프"""

    def __init__(self, base_trainer, history_path=None):
        self.base_trainer = base_trainer
        self.history_path = Path(history_path) if history_path else None
        self.history = []
        self.adjuster = RewardAdjuster()
        self.stop_requested = False

    def _converged(self, metrics, target_sr, target_cr):
        return (metrics.get("success_rate",0) >= target_sr and
                metrics.get("collision_rate",1) <= target_cr)

    def _save_history(self):
        if self.history_path is None: return
        self.history_path.parent.mkdir(parents=True, exist_ok=True)
        data = _clean_nan([dataclasses.asdict(r) for r in self.history])
        self.history_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

    def run(self, max_iterations=5, target_success_rate=0.70, target_collision_rate=0.15,
            eval_episodes=50, eval_difficulty="medium", base_timesteps=300_000):
        current_weights = RewardWeights()

        for i in range(1, max_iterations + 1):
            if self.stop_requested: break
            logger.info(f"[Iterative] ===== 반복 {i}/{max_iterations} =====")

            # 사전 평가 (모델 있을 때만)
            pre_metrics = {}
            if self.base_trainer.agent.model is not None and i > 1:
                pre_metrics = self.base_trainer.evaluate(eval_episodes, eval_difficulty)
                logger.info(f"[Iterative] 사전평가: {pre_metrics}")
                if self._converged(pre_metrics, target_success_rate, target_collision_rate):
                    logger.info("[Iterative] 수렴 달성 — 조기 종료")
                    break

            # 커리큘럼 학습
            logger.info(f"[Iterative] 커리큘럼 학습 시작 (weights: collision={current_weights.collision:.1f}, progress={current_weights.progress:.1f}, success={current_weights.episode_success:.1f})")
            self.base_trainer.train_curriculum(reward_weights=current_weights, base_timesteps=base_timesteps)

            # 사후 평가
            post_metrics = self.base_trainer.evaluate(eval_episodes, eval_difficulty)
            logger.info(f"[Iterative] 사후평가: {post_metrics}")

            # 시그널 분석
            signals = self.adjuster.analyze(post_metrics)
            history_dicts = [dataclasses.asdict(r) for r in self.history]
            col_plateau = ("critical_collision" in signals or "high_collision" in signals) and \
                          self.adjuster.check_plateau(history_dicts, "collision_rate")
            suc_plateau = "low_success" in signals and \
                          self.adjuster.check_plateau(history_dicts, "success_rate")

            if col_plateau and suc_plateau:
                logger.info("[Iterative] 완전 Plateau — 가중치 리셋")
                next_weights = RewardWeights(collision=-500.0, proximity=-10.0, danger_zone=-20.0,
                    route_deviation=-0.2, progress=3.0, smoothness=-0.05, fuel=-0.02,
                    ice_concentration=-0.3, episode_success=500.0)
            elif col_plateau:
                logger.info("[Iterative] Collision Plateau — danger_zone 집중 강화")
                d = dataclasses.asdict(current_weights)
                d["danger_zone"] *= 2.5; d["proximity"] *= 1.5
                next_weights = _apply_bounds(RewardWeights(**d))
            elif suc_plateau:
                logger.info("[Iterative] Success Plateau — episode_success 집중 강화")
                d = dataclasses.asdict(current_weights)
                d["episode_success"] *= 2.0; d["progress"] *= 2.0
                next_weights = _apply_bounds(RewardWeights(**d))
            else:
                next_weights = self.adjuster.adjust(current_weights, signals)

            converged = self._converged(post_metrics, target_success_rate, target_collision_rate)
            record = IterationRecord(
                iteration=i, weights=dataclasses.asdict(current_weights),
                pre_metrics=pre_metrics, post_metrics=post_metrics,
                signals=signals, duration_seconds=0.0, converged=converged)
            self.history.append(record)
            self._save_history()

            logger.info(f"[Iterative] 반복 {i} 완료 | success={post_metrics.get(\'success_rate\',0):.3f} | "
                        f"collision={post_metrics.get(\'collision_rate\',0):.3f} | signals={signals} | converged={converged}")

            if converged:
                logger.info("[Iterative] 수렴 완료")
                break
            current_weights = next_weights

        final = self.history[-1].post_metrics if self.history else {}
        return {"iterations": len(self.history),
                "converged": self.history[-1].converged if self.history else False,
                "final_metrics": final}
''')
print('트레이너 + IterativeTrainer 생성 완료')

In [ ]:
# ============================================================
# CELL 8: 모듈 검증
# ============================================================
import sys
if '/content/arctic' not in sys.path:
    sys.path.insert(0, '/content/arctic')

from modules.rl_environment import IcebergAvoidanceEnv
from modules.rl_trainer import RLTrainer, IterativeTrainer, RewardAdjuster
from modules.rl_reward import RewardWeights

env = IcebergAvoidanceEnv(difficulty='easy')
obs, _ = env.reset()
obs2, r, term, trunc, info = env.step(env.action_space.sample())
print(f'환경 OK: obs={obs.shape}, reward={r:.2f}')

adj = RewardAdjuster()
test_signals = adj.analyze({'collision_rate': 0.3, 'success_rate': 0.1})
print(f'RewardAdjuster OK: signals={test_signals}')
print('모든 모듈 검증 완료')

In [ ]:
# ============================================================
# CELL 9: 학습 설정
# ============================================================
import os

ROUTES      = ['NSR', 'NWP', 'TSR']
ICE_CLASSES = ['PC7', 'PC6', 'PC5', 'PC4', 'PC3', 'IA Super', 'IA']
SHIP_TYPES  = {
    'bulk':      {'max_speed_knots': 12.0, 'ice_drag_factor': 0.50},
    'tanker':    {'max_speed_knots': 14.0, 'ice_drag_factor': 0.45},
    'container': {'max_speed_knots': 18.0, 'ice_drag_factor': 0.35},
    'lng':       {'max_speed_knots': 16.0, 'ice_drag_factor': 0.40},
}

# ---- 반복 학습 설정 ----
BASE_TIMESTEPS   = 300_000   # 커리큘럼 총 타임스텝
MAX_ITERATIONS   = 5         # 최대 반복 횟수 (수렴 시 조기 종료)
EVAL_EPISODES    = 50        # 평가 에피소드 수
EVAL_DIFFICULTY  = 'medium'  # 평가 난이도
TARGET_SUCCESS   = 0.70      # 수렴 기준: 성공률
TARGET_COLLISION = 0.15      # 수렴 기준: 충돌률

MODEL_BASE_DIR   = '/content/arctic/models'
DRIVE_MODEL_DIR  = f'{DRIVE_DIR}/models'
DRIVE_LOG_DIR    = f'{DRIVE_DIR}/logs'
os.makedirs(MODEL_BASE_DIR, exist_ok=True)

# colab_train.ipynb에서 학습한 모델이 Drive에 있으면 로컬로 복사
import shutil
if os.path.exists(DRIVE_MODEL_DIR):
    for model_dir in os.listdir(DRIVE_MODEL_DIR):
        src = f'{DRIVE_MODEL_DIR}/{model_dir}'
        dst = f'{MODEL_BASE_DIR}/{model_dir}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
    print(f'Drive에서 모델 복사 완료: {len(os.listdir(MODEL_BASE_DIR))}개')
else:
    print('Drive 모델 없음 — 처음부터 학습')

ALL_COMBOS = [(r, ic, st) for r in ROUTES for ic in ICE_CLASSES for st in SHIP_TYPES]
print(f'학습 조합: {len(ALL_COMBOS)}개')
print(f'최대 반복: {MAX_ITERATIONS}회 / 수렴 기준: success≥{TARGET_SUCCESS}, collision≤{TARGET_COLLISION}')

In [ ]:
# ============================================================
# CELL 10: 반복 학습 실행 (보상 자동조정 포함)
# ============================================================
import logging, time, json, shutil
from datetime import datetime
from pathlib import Path
from modules.rl_trainer import RLTrainer, IterativeTrainer
from modules.rl_ship_dynamics import ShipParams

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[
        logging.FileHandler(f'{DRIVE_LOG_DIR}/iterative_{datetime.now().strftime("%Y%m%d_%H%M")}.log'),
        logging.StreamHandler(),
    ]
)
logger = logging.getLogger('colab_iterative')

results_path = f'{DRIVE_LOG_DIR}/iterative_results.json'
all_results = json.loads(Path(results_path).read_text()) if Path(results_path).exists() else {}
logger.info(f'이전 결과: {len(all_results)}개')

total = len(ALL_COMBOS)
done = 0
start_all = time.time()

for route, ice_class, ship_type in ALL_COMBOS:
    combo_key = f'{route}_{ice_class}_{ship_type}'.replace(' ', '_')

    # 수렴 완료된 조합 스킵
    if combo_key in all_results and all_results[combo_key].get('converged'):
        done += 1
        logger.info(f'[{done}/{total}] SKIP (수렴): {combo_key}')
        continue

    ship_cfg = SHIP_TYPES[ship_type]
    ship_params = ShipParams(max_speed_knots=ship_cfg['max_speed_knots'],
                             ice_drag_factor=ship_cfg['ice_drag_factor'])

    base_trainer = RLTrainer(
        model_key=combo_key,
        fixed_route=route,
        fixed_ice_class=ice_class,
        ship_params=ship_params,
        model_base_dir=MODEL_BASE_DIR,
    )

    # 기존 모델 로드 (colab_train.ipynb에서 학습한 모델 이어받기)
    if base_trainer.agent.load():
        logger.info(f'[{done+1}/{total}] 기존 모델 로드: {combo_key}')
    else:
        logger.info(f'[{done+1}/{total}] 새 모델: {combo_key}')

    history_path = f'{DRIVE_LOG_DIR}/history_{combo_key}.json'
    iterative = IterativeTrainer(
        base_trainer=base_trainer,
        history_path=history_path,
    )

    logger.info(f'[{done+1}/{total}] 반복 학습 시작: {combo_key}')
    t0 = time.time()

    result = iterative.run(
        max_iterations=MAX_ITERATIONS,
        target_success_rate=TARGET_SUCCESS,
        target_collision_rate=TARGET_COLLISION,
        eval_episodes=EVAL_EPISODES,
        eval_difficulty=EVAL_DIFFICULTY,
        base_timesteps=BASE_TIMESTEPS,
    )

    elapsed = time.time() - t0
    all_results[combo_key] = {
        'route': route, 'ice_class': ice_class, 'ship_type': ship_type,
        'converged': result.get('converged', False),
        'iterations': result.get('iterations', 0),
        'final_metrics': result.get('final_metrics', {}),
        'elapsed_sec': elapsed,
    }

    # Drive에 결과 저장
    Path(results_path).write_text(json.dumps(all_results, indent=2, ensure_ascii=False))

    # 모델을 Drive에 복사
    src = f'{MODEL_BASE_DIR}/sac_{combo_key}'
    dst = f'{DRIVE_MODEL_DIR}/sac_{combo_key}'
    if Path(src).exists():
        if Path(dst).exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)

    done += 1
    elapsed_all = (time.time() - start_all) / 3600
    eta = elapsed_all / done * (total - done) if done > 0 else 0
    conv = sum(1 for v in all_results.values() if v.get('converged'))
    logger.info(f'[{done}/{total}] 완료: {combo_key} | 수렴={result.get("converged")} | '
                f'전체수렴={conv}/{total} | 경과={elapsed_all:.1f}h | ETA={eta:.1f}h')

conv_total = sum(1 for v in all_results.values() if v.get('converged'))
logger.info(f'=== 전체 완료: {done}/{total}개, 수렴={conv_total}개 ===')

In [ ]:
# ============================================================
# CELL 11: 결과 요약
# ============================================================
import json
from pathlib import Path

results_path = f'{DRIVE_DIR}/logs/iterative_results.json'
all_results = json.loads(Path(results_path).read_text()) if Path(results_path).exists() else {}

total = len(all_results)
converged = sum(1 for v in all_results.values() if v.get('converged'))
print(f'완료: {total}개 / 수렴: {converged}개')
print()
print(f'{"조합":<35} {"수렴":>6} {"반복":>4} {"성공률":>8} {"충돌률":>8}')
print('-'*65)
for key, v in sorted(all_results.items()):
    m = v.get('final_metrics', {})
    print(f'{key:<35} {str(v.get("converged",False)):>6} {v.get("iterations",0):>4} '
          f'{m.get("success_rate",0):>8.3f} {m.get("collision_rate",0):>8.3f}')

## 사용 가이드

### 실행 순서
1. 런타임 > 런타임 유형 변경 > **T4 GPU**
2. **셀 1~10 순서대로 실행**

### colab_train.ipynb 이후 이어받기
- Drive > `arctic_rl/models/` 에 기존 모델이 있으면 **자동으로 로드**
- 로드된 모델부터 반복 학습 재개

### 반복 학습 흐름
```
로드 → 평가 → 시그널 분석 → 가중치 조정 → 재학습 → 평가 → ...
          ↑_________________________________|
                success≥0.70 & collision≤0.15 이면 수렴 종료
```

### 보상 가중치 자동조정 규칙
| 시그널 | 조치 |
|---|---|
| critical_collision (충돌>20%) | collision×2, proximity×1.8, danger_zone×2 |
| high_collision (충돌>10%) | collision×1.6, proximity×1.4 |
| low_success (성공<60%) | episode_success×1.8, progress×1.5 |
| high_deviation | route_deviation×1.5 |
| Plateau 감지 | 가중치 리셋 또는 집중 강화 |

### 예상 시간 (T4 GPU)
| | |
|--|--|
| 조합당 1 iteration | ~2~3분 (학습+평가) |
| 84조합 × 5 iterations | ~14~21시간 |
| 세션 재개 시 | 수렴된 조합 자동 스킵 |